# 🧪 NLP Evaluation Studio

Interactive evaluation notebook untuk mengevaluasi respons backend dengan metrik NLP custom.

**Features:**
- ✅ Custom system prompt input
- ✅ Learning profile (task, persona, mission objective)
- ✅ Specific prompt dari database
- ✅ Model selection
- ✅ Tool calling (refine_prompt, semantic_search)
- ✅ NLP metrics (BLEU, ROUGE-L, METEOR, BERTScore)
- ✅ Reference answer comparison

## Setup & Imports

In [1]:
import os
import sys
import json
import asyncio
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, asdict
import numpy as np
from IPython.display import display, Markdown, HTML

# Add backend to path
sys.path.insert(0, str(Path.cwd()))

# Load environment
from dotenv import load_dotenv
load_dotenv('.env.local')

# Metrics imports
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score_fn
from openai import OpenAI

# Download NLTK data
try:
    import nltk
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading NLTK data...")
    nltk.download('punkt', quiet=True)
    nltk.download('wordnet', quiet=True)

print("✓ All imports loaded successfully")

✓ All imports loaded successfully


## Database Connection & Helper Functions

In [2]:
class DatabaseConnection:
    """Docker PostgreSQL connection via docker-compose exec"""
    def __init__(self, working_dir: str = "."):
        self.working_dir = working_dir
        self.container_name = "system-llm-postgres-local"
        self.user = "llm_user"
        self.db = "system_llm"

    def execute_sql(self, query: str, fetch: bool = False) -> str:
        """Execute SQL via docker-compose exec"""
        cmd = [
            "docker-compose", "-f", "docker-compose.local.yml",
            "exec", "-T", "postgres",
            "psql", "-U", self.user, "-d", self.db,
        ]
        
        if fetch:
            cmd.append("-t")

        try:
            result = subprocess.run(
                cmd,
                input=query,
                capture_output=True,
                text=True,
                encoding='utf-8',
                errors='replace',
                cwd=self.working_dir
            )

            if result.returncode != 0:
                error_msg = result.stderr.strip() if result.stderr else "Unknown error"
                raise Exception(f"SQL Error: {error_msg}")

            return result.stdout.strip() if fetch else ""
        except Exception as e:
            print(f"Database Error: {e}")
            raise

    def test_connection(self) -> bool:
        try:
            version = self.execute_sql("SELECT version();", fetch=True)
            if version:
                db_version = version.split(',')[0].strip()
                print(f"✓ Connected to {self.container_name}")
                print(f"  Version: {db_version}")
                return True
        except Exception as e:
            print(f"✗ Connection failed: {e}")
            return False

# Initialize database
db = DatabaseConnection(working_dir=".")
print("="*70)
print("DATABASE CONNECTION TEST")
print("="*70)

if not db.test_connection():
    sys.exit(1)

# OpenAI client
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("ERROR: OPENAI_API_KEY not set in .env.local")
    sys.exit(1)

client = OpenAI(api_key=OPENAI_API_KEY)
print("✓ OpenAI API: Ready\n")

DATABASE CONNECTION TEST
✓ Connected to system-llm-postgres-local
  Version: PostgreSQL 15.15 (Debian 15.15-1.pgdg12+1) on x86_64-pc-linux-gnu
✓ OpenAI API: Ready



## Load Available Models & Data

In [3]:
def list_models() -> List[Dict[str, Any]]:
    """Query database for available models"""
    try:
        query = "SELECT id, name, provider, \"order\" FROM model ORDER BY \"order\" ASC;"
        result = db.execute_sql(query, fetch=True)
        
        model_list = []
        if result:
            for line in result.split('\n'):
                if '|' in line and not line.startswith('-'):
                    parts = line.split('|')
                    if len(parts) >= 4:
                        try:
                            model_id = parts[0].strip()
                            name = parts[1].strip()
                            provider = parts[2].strip()
                            order = parts[3].strip()
                            
                            if model_id and name:
                                model_list.append({
                                    "id": model_id,
                                    "name": name,
                                    "provider": provider,
                                    "order": order,
                                })
                        except:
                            pass
        
        return model_list
    except Exception as e:
        print(f"Error listing models: {e}")
        return []

def list_prompts() -> List[Dict[str, Any]]:
    """Query database for available prompts"""
    try:
        query = "SELECT id, content, is_active FROM prompt ORDER BY created_at DESC;"
        result = db.execute_sql(query, fetch=True)
        
        prompt_list = []
        if result:
            for line in result.split('\n'):
                if '|' in line and not line.startswith('-'):
                    parts = line.split('|', 2)  # Split only on first 2 pipes
                    if len(parts) >= 3:
                        try:
                            prompt_id = parts[0].strip()
                            content = parts[1].strip()[:100]  # First 100 chars
                            is_active = parts[2].strip()
                            
                            if prompt_id:
                                prompt_list.append({
                                    "id": prompt_id,
                                    "content_preview": content,
                                    "is_active": is_active == "t",
                                })
                        except:
                            pass
        
        return prompt_list
    except Exception as e:
        print(f"Error listing prompts: {e}")
        return []

def get_prompt_content(prompt_id: str) -> Optional[str]:
    """Get full prompt content from database"""
    try:
        query = f"SELECT content FROM prompt WHERE id = '{prompt_id}' LIMIT 1;"
        result = db.execute_sql(query, fetch=True)
        
        if result:
            for line in result.split('\n'):
                if line and not line.startswith('-'):
                    return line.strip()
        return None
    except Exception as e:
        print(f"Error getting prompt: {e}")
        return None

def get_chat_config() -> Dict[str, Any]:
    """Get ChatConfig singleton from database"""
    try:
        query = """SELECT id, prompt_general, prompt_refine, default_top_k, 
                          max_top_k, similarity_threshold, tool_calling_max_iterations
                   FROM chat_config WHERE id = 1;"""
        result = db.execute_sql(query, fetch=True)
        
        if result:
            for line in result.split('\n'):
                if '|' in line and not line.startswith('-'):
                    parts = line.split('|')
                    if len(parts) >= 7:
                        return {
                            "prompt_general": parts[1].strip() if parts[1].strip() else None,
                            "prompt_refine": parts[2].strip() if parts[2].strip() else "Refine the student's question",
                            "default_top_k": int(parts[3].strip()) if parts[3].strip() else 5,
                            "max_top_k": int(parts[4].strip()) if parts[4].strip() else 10,
                            "similarity_threshold": float(parts[5].strip()) if parts[5].strip() else 0.7,
                            "tool_calling_max_iterations": int(parts[6].strip()) if parts[6].strip() else 10,
                        }
        
        return {
            "prompt_general": None,
            "prompt_refine": "Refine the student's question to be more specific and clear.",
            "default_top_k": 5,
            "max_top_k": 10,
            "similarity_threshold": 0.7,
            "tool_calling_max_iterations": 10,
        }
    except Exception as e:
        print(f"Warning: Could not get ChatConfig: {e}")
        return {
            "prompt_general": None,
            "prompt_refine": "Refine the student's question to be more specific and clear.",
            "default_top_k": 5,
            "max_top_k": 10,
            "similarity_threshold": 0.7,
            "tool_calling_max_iterations": 10,
        }

# Load data
available_models = list_models()
available_prompts = list_prompts()
chat_config = get_chat_config()

print(f"\n📊 Available Models: {len(available_models)}")
for m in available_models:
    print(f"  - {m['name']} ({m['provider']})")

print(f"\n📋 Available Prompts: {len(available_prompts)}")
for p in available_prompts[:5]:  # Show first 5
    print(f"  - {p['id']}: {p['content_preview']}...")


📊 Available Models: 6
  - gpt-4.1-nano (OPENAI)
  - claude-haiku-4-5 (ANTHROPIC)
  - gemini-2.5-flash (GOOGLE)
  - meta-llama/llama-3.1-8b-instruct (openrouter)
  - qwen/qwen-2.5-7b-instruct (openrouter)
  - microsoft/phi-3-medium-128k-instruct (openrouter)

📋 Available Prompts: 2
  - 3f2480db-7cc1-42ea-a7c9-357d2ca8a49b: ini prompt kedua...
  - d79762f5-c06b-451a-b614-6ce74915aa74: harus import library...


## Core Evaluation Components

In [4]:
@dataclass
class EvaluationConfig:
    """Configuration for evaluation run"""
    # LLM Selection
    model_name: str = "gpt-4.1-nano"
    temperature: float = 0.7

    # Tool Configuration
    use_tools: bool = True
    max_tool_iterations: int = 3

    # System Prompt Overrides (in priority order)
    prompt_general: Optional[str] = None  # Highest priority - direct input

    # Learning Profile (mid priority)
    task: Optional[str] = None
    persona: Optional[str] = None
    mission_objective: Optional[str] = None

    # Specific Prompt (lowest priority) - can be direct input or from DB
    prompt_specific: Optional[str] = None  # Direct input (NEW)
    prompt_id: Optional[str] = None  # From database (legacy)

    # RAG Configuration
    with_rag: bool = True
    top_k: Optional[int] = None
    similarity_threshold: Optional[float] = None

class PromptAssembler:
    """Assemble system prompt following backend priority order"""

    @staticmethod
    def assemble(config: EvaluationConfig, chat_config: Dict[str, Any]) -> Tuple[str, Dict[str, Any]]:
        """
        Assemble system prompt in backend priority order:
        1. prompt_general (HIGHEST PRIORITY)
        2. Learning Profile (task, persona, mission_objective)
        3. Specific Prompt (LOWEST PRIORITY) - can be direct input or from DB

        Returns: (final_system_prompt, prompt_sources)
        """
        sections = []
        sources = {}

        # SECTION 1: General Prompt (HIGHEST PRIORITY)
        prompt_general = config.prompt_general
        if not prompt_general:
            prompt_general = chat_config.get("prompt_general")

        if prompt_general:
            sections.append(f"# General Prompt\n{prompt_general}")
            sources["prompt_general"] = prompt_general[:100] + "..." if len(prompt_general) > 100 else prompt_general

        # SECTION 2: Learning Profile (MID PRIORITY)
        profile_sections = []

        if config.task:
            profile_sections.append(f"# Task\n{config.task}")
            sources["task"] = config.task[:80] + "..." if len(config.task) > 80 else config.task

        if config.persona:
            profile_sections.append(f"# Persona\n{config.persona}")
            sources["persona"] = config.persona[:80] + "..." if len(config.persona) > 80 else config.persona

        if config.mission_objective:
            profile_sections.append(f"# Mission Objective\n{config.mission_objective}")
            sources["mission_objective"] = config.mission_objective[:80] + "..." if len(config.mission_objective) > 80 else config.mission_objective

        if profile_sections:
            sections.append("## Student Learning Profile\n" + "\n\n".join(profile_sections))

        # SECTION 3: Specific Prompt (LOWEST PRIORITY)
        # Priority: direct input > from database
        prompt_specific = config.prompt_specific
        if not prompt_specific and config.prompt_id:
            prompt_specific = get_prompt_content(config.prompt_id)

        if prompt_specific:
            sections.append(f"# Specific Prompt\n{prompt_specific}")
            sources["specific_prompt"] = prompt_specific[:100] + "..." if len(prompt_specific) > 100 else prompt_specific
            if config.prompt_id:
                sources["specific_prompt_source"] = f"From DB: {config.prompt_id}"
            else:
                sources["specific_prompt_source"] = "Direct input"

        final_prompt = "\n\n".join(sections) if sections else "You are a helpful educational assistant."

        return final_prompt, sources

print("✓ Core components ready")

✓ Core components ready


## RAG & Tool Execution

In [5]:
def generate_embedding(text: str) -> List[float]:
    """Generate embedding using OpenAI API"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def semantic_search(query_text: str, top_k: int = 5, similarity_threshold: float = 0.7) -> List[Dict[str, Any]]:
    """Semantic search with similarity threshold filter"""
    try:
        query_embedding = np.array(generate_embedding(query_text))
        
        # Get document chunks from database
        search_query = """
        SELECT dc.content, d.original_filename, dc.page_number, dc.embedding
        FROM document_chunk dc
        JOIN document d ON dc.document_id = d.id
        WHERE d.status = 'PROCESSED'
        LIMIT 100;
        """
        result_text = db.execute_sql(search_query, fetch=True)
        
        similarities = []
        if result_text:
            for line in result_text.split('\n'):
                if '|' in line:
                    parts = line.split('|', 3)
                    if len(parts) >= 4:
                        try:
                            content = parts[0].strip()
                            filename = parts[1].strip()
                            page_num = int(parts[2].strip()) if parts[2].strip().isdigit() else 0
                            embedding_json = parts[3].strip()
                            
                            if embedding_json.startswith('['):
                                chunk_embedding = np.array(json.loads(embedding_json))
                                similarity = np.dot(query_embedding, chunk_embedding) / (
                                    np.linalg.norm(query_embedding) * np.linalg.norm(chunk_embedding) + 1e-10
                                )
                                
                                # APPLY SIMILARITY THRESHOLD (production behavior)
                                if similarity >= similarity_threshold:
                                    similarities.append({
                                        "content": content[:500],
                                        "filename": filename,
                                        "page_number": page_num,
                                        "similarity_score": float(similarity)
                                    })
                        except:
                            pass
        
        similarities.sort(key=lambda x: x['similarity_score'], reverse=True)
        return similarities[:top_k]
    
    except Exception as e:
        print(f"Semantic search error: {e}")
        return []

class ToolExecutor:
    """Execute tools during LLM calls"""
    
    def __init__(self, chat_config: Dict[str, Any]):
        self.chat_config = chat_config
    
    def get_tools_definition(self) -> List[Dict[str, Any]]:
        """Get tool definitions for LLM"""
        return [
            {
                "name": "refine_prompt",
                "description": "Refine and clarify an ambiguous or unclear student question",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "original_prompt": {"type": "string", "description": "The original student question"},
                    },
                    "required": ["original_prompt"]
                }
            },
            {
                "name": "semantic_search",
                "description": "Search for relevant documents based on semantic similarity",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query"},
                        "top_k": {"type": "integer", "description": "Number of results", "default": 5},
                    },
                    "required": ["query"]
                }
            }
        ]
    
    async def execute_refine_prompt(self, args: Dict[str, Any], verbose: bool = False) -> Dict[str, Any]:
        """Execute refine_prompt tool"""
        original_prompt = args.get("original_prompt", "")
        
        if not original_prompt:
            return {"original": original_prompt, "refined": original_prompt, "success": False, "error": "Empty prompt"}
        
        try:
            refine_instruction = self.chat_config.get("prompt_refine")
            
            if verbose:
                print(f"\n      [refine_prompt] Instruction: {refine_instruction[:80]}...")
            
            response = client.chat.completions.create(
                model="gpt-4.1-nano",
                messages=[
                    {"role": "system", "content": refine_instruction},
                    {"role": "user", "content": original_prompt}
                ],
                temperature=0.7,
                max_tokens=300
            )
            
            refined_prompt = response.choices[0].message.content or original_prompt
            
            if verbose:
                print(f"      Original: {original_prompt[:60]}...")
                print(f"      Refined:  {refined_prompt[:60]}...")
            
            return {
                "original": original_prompt,
                "refined": refined_prompt,
                "success": True
            }
        except Exception as e:
            return {
                "original": original_prompt,
                "refined": original_prompt,
                "success": False,
                "error": str(e)
            }
    
    async def execute_semantic_search(self, args: Dict[str, Any], verbose: bool = False) -> Dict[str, Any]:
        """Execute semantic_search tool"""
        query = args.get("query", "")
        top_k = args.get("top_k", self.chat_config.get("default_top_k", 5))
        similarity_threshold = self.chat_config.get("similarity_threshold", 0.7)
        
        try:
            results = semantic_search(query, top_k=top_k, similarity_threshold=similarity_threshold)
            
            if verbose:
                print(f"\n      [semantic_search] Query: {query}")
                print(f"      Found {len(results)} documents (threshold: {similarity_threshold})")
                for i, r in enumerate(results[:3], 1):
                    print(f"        [{i}] {r['filename']} (similarity: {r['similarity_score']:.3f})")
            
            return {
                "query": query,
                "results": [
                    {
                        "content": r["content"],
                        "filename": r["filename"],
                        "page": r["page_number"],
                        "similarity": r["similarity_score"],
                    }
                    for r in results
                ]
            }
        except Exception as e:
            return {"error": str(e), "query": query}

print("✓ RAG & Tool execution ready")

✓ RAG & Tool execution ready


## Chat Service with Tool Calling

In [6]:
class ChatServiceWithTools:
    """Chat service with agentic tool calling"""
    
    def __init__(self, chat_config: Dict[str, Any]):
        self.chat_config = chat_config
        self.tool_executor = ToolExecutor(chat_config)
    
    async def send_message(
        self,
        user_message: str,
        system_prompt: str,
        model_name: str,
        temperature: float = 0.7,
        use_tools: bool = True,
        max_iterations: int = 3,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """Send message with optional tool calling"""
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
        
        tools_def = self.tool_executor.get_tools_definition() if use_tools else None
        tool_calls_made = []
        iteration = 0
        
        while iteration < max_iterations:
            iteration += 1
            
            if verbose:
                print(f"  [Iteration {iteration}/{max_iterations}]", end=" ")
            
            # Call LLM
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                tools=[{"type": "function", "function": t} for t in tools_def] if tools_def else None,
                tool_choice="auto" if tools_def else None,
                temperature=temperature
            )
            
            assistant_message = response.choices[0].message
            
            if assistant_message.tool_calls:
                # Process tool calls
                messages.append({
                    "role": "assistant",
                    "content": assistant_message.content or "",
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments
                            }
                        }
                        for tc in assistant_message.tool_calls
                    ]
                })
                
                # Execute tools
                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name
                    
                    if verbose:
                        print(f"🔧 {tool_name}", end=" ")
                    
                    try:
                        tool_args = json.loads(tool_call.function.arguments)
                        
                        if tool_name == "refine_prompt":
                            result = await self.tool_executor.execute_refine_prompt(tool_args, verbose=verbose)
                        elif tool_name == "semantic_search":
                            result = await self.tool_executor.execute_semantic_search(tool_args, verbose=verbose)
                        else:
                            result = {"error": f"Unknown tool: {tool_name}"}
                        
                        # Add tool result to messages
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": json.dumps(result)
                        })
                        
                        tool_calls_made.append({
                            "name": tool_name,
                            "args": tool_args,
                            "result": result
                        })
                        
                        if verbose:
                            print("✓", end=" ")
                    except Exception as e:
                        if verbose:
                            print(f"✗ {str(e)[:30]}", end=" ")
                
                if verbose:
                    print()
            else:
                # No more tool calls, return response
                if verbose:
                    print("✓ Done")
                
                return {
                    "content": assistant_message.content or "",
                    "tool_calls": tool_calls_made,
                    "iterations": iteration
                }
        
        # Max iterations reached
        if verbose:
            print("⚠ Max iterations reached")
        
        return {
            "content": assistant_message.content or "",
            "tool_calls": tool_calls_made,
            "iterations": iteration
        }

print("✓ Chat service ready")

✓ Chat service ready


## NLP Evaluation Metrics

In [7]:
class EvaluationMetrics:
    """NLP evaluation metrics - with IndoBERT support for Indonesian text"""
    
    @staticmethod
    def bleu_score(reference: str, candidate: str, weights: tuple = (0.25, 0.25, 0.25, 0.25)) -> float:
        try:
            reference_tokens = word_tokenize(reference.lower())
            candidate_tokens = word_tokenize(candidate.lower())
            smoothing_function = SmoothingFunction().method1
            score = sentence_bleu(
                [reference_tokens],
                candidate_tokens,
                weights=weights,
                smoothing_function=smoothing_function
            )
            return float(score)
        except Exception as e:
            return 0.0
    
    @staticmethod
    def rouge_l_score(reference: str, candidate: str) -> float:
        try:
            scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
            scores = scorer.score(reference, candidate)
            return float(scores['rougeL'].fmeasure)
        except Exception as e:
            return 0.0
    
    @staticmethod
    def meteor_score(reference: str, candidate: str) -> float:
        try:
            reference_tokens = word_tokenize(reference.lower())
            candidate_tokens = word_tokenize(candidate.lower())
            score = meteor_score([reference_tokens], candidate_tokens)
            return float(score)
        except Exception as e:
            return 0.0
    
    @staticmethod
    def bert_score(reference: str, candidate: str, model_type: str = "indobert-base-p1") -> Dict[str, float]:
        """
        Calculate BERTScore using IndoBERT for Indonesian text
        model_type options:
        - "indobert-base-p1" (DEFAULT - best for Indonesian)
        - "indobert-large-p1"
        - "distilbert-base-uncased" (fallback for English)
        """
        try:
            P, R, F1 = bert_score_fn([candidate], [reference], model_type=model_type, verbose=False)
            return {
                "precision": float(P[0]),
                "recall": float(R[0]),
                "f1": float(F1[0]),
                "model": model_type
            }
        except Exception as e:
            print(f"    ⚠ BERTScore ({model_type}) failed: {str(e)[:50]}. Using fallback...")
            try:
                # Fallback to English model
                P, R, F1 = bert_score_fn([candidate], [reference], model_type="distilbert-base-uncased", verbose=False)
                return {
                    "precision": float(P[0]),
                    "recall": float(R[0]),
                    "f1": float(F1[0]),
                    "model": "distilbert-base-uncased (fallback)"
                }
            except:
                return {
                    "precision": 0.0,
                    "recall": 0.0,
                    "f1": 0.0,
                    "model": "FAILED"
                }
    
    @staticmethod
    def calculate_all(reference: str, candidate: str, bert_model: str = "indobert-base-p1") -> Dict[str, Any]:
        """
        Calculate all metrics
        
        Args:
            reference: Reference answer
            candidate: Generated answer
            bert_model: BERT model to use (default: indobert-base-p1 for Indonesian)
        """
        print("  📊 Computing metrics:", end=" ")
        
        metrics = {}
        
        print("BLEU", end=" ")
        metrics["bleu"] = EvaluationMetrics.bleu_score(reference, candidate)
        
        print("ROUGE-L", end=" ")
        metrics["rouge_l"] = EvaluationMetrics.rouge_l_score(reference, candidate)
        
        print("METEOR", end=" ")
        metrics["meteor"] = EvaluationMetrics.meteor_score(reference, candidate)
        
        print("BERTScore", end=" ")
        metrics["bert_score"] = EvaluationMetrics.bert_score(reference, candidate, model_type=bert_model)
        
        # Length ratio
        ref_len = len(reference.split())
        cand_len = len(candidate.split())
        metrics["length_ratio"] = cand_len / ref_len if ref_len > 0 else 0.0
        
        print("✓")
        
        return metrics

print("✓ Evaluation metrics ready (with IndoBERT support)")

✓ Evaluation metrics ready (with IndoBERT support)


## Main Evaluation Function

In [8]:
async def run_evaluation(
    config: EvaluationConfig,
    user_question: str,
    reference_answer: str,
    bert_model: str = "indobert-base-p1",
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run complete evaluation pipeline:
    1. Assemble system prompt
    2. Send message with tool calling
    3. Calculate metrics with specified BERT model
    
    Args:
        config: EvaluationConfig with all settings
        user_question: User's question to evaluate
        reference_answer: Reference answer for comparison
        bert_model: BERT model to use (default: indobert-base-p1 for Indonesian)
                   Options: "indobert-base-p1", "indobert-large-p1", "distilbert-base-uncased"
        verbose: Show detailed output
    """
    
    print(f"\n{'='*80}")
    print(f"🧪 NLP EVALUATION RUN")
    print(f"{'='*80}")
    print(f"Model: {config.model_name}")
    print(f"With Tools: {config.use_tools}")
    print(f"Temperature: {config.temperature}")
    print(f"Max Iterations: {config.max_tool_iterations}")
    print(f"BERT Model: {bert_model}")
    print(f"{'-'*80}")
    
    # Assemble system prompt
    system_prompt, prompt_sources = PromptAssembler.assemble(config, chat_config)
    
    if verbose:
        print(f"\n📋 SYSTEM PROMPT SOURCES:")
        for key, value in prompt_sources.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in value.items():
                    print(f"    - {k}: {v}")
            else:
                print(f"  - {key}: {value}")
        
        print(f"\n📝 ASSEMBLED SYSTEM PROMPT:")
        print(f"{'-'*80}")
        print(system_prompt[:500] + ("..." if len(system_prompt) > 500 else ""))
        print(f"{'-'*80}")
    
    print(f"\n🤖 SENDING MESSAGE TO LLM...")
    print(f"  Question: {user_question[:80]}..." if len(user_question) > 80 else f"  Question: {user_question}")
    
    # Send message with tools
    chat_service = ChatServiceWithTools(chat_config)
    response = await chat_service.send_message(
        user_message=user_question,
        system_prompt=system_prompt,
        model_name=config.model_name,
        temperature=config.temperature,
        use_tools=config.use_tools,
        max_iterations=config.max_tool_iterations,
        verbose=verbose
    )
    
    response_text = response.get("content", "")
    tool_calls = response.get("tool_calls", [])
    iterations = response.get("iterations", 0)
    
    print(f"\n✓ Response received ({len(response_text)} chars)")
    print(f"  Iterations: {iterations}")
    print(f"  Tool calls: {len(tool_calls)}")
    
    if tool_calls and verbose:
        print(f"\n🔧 TOOL CALLS MADE:")
        for i, tc in enumerate(tool_calls, 1):
            print(f"  [{i}] {tc['name']}")
    
    # Display response
    print(f"\n💬 LLM RESPONSE:")
    print(f"{'-'*80}")
    print(response_text[:600] + ("..." if len(response_text) > 600 else ""))
    print(f"{'-'*80}")
    
    # Calculate metrics with specified BERT model
    print(f"\n📊 CALCULATING METRICS...")
    metrics = EvaluationMetrics.calculate_all(reference_answer, response_text, bert_model=bert_model)
    
    # Display metrics
    print(f"\n📈 EVALUATION RESULTS:")
    print(f"{'-'*80}")
    print(f"BLEU Score:              {metrics['bleu']:.4f}")
    print(f"ROUGE-L Score:           {metrics['rouge_l']:.4f}")
    print(f"METEOR Score:            {metrics['meteor']:.4f}")
    print(f"BERTScore (F1):          {metrics['bert_score']['f1']:.4f}")
    print(f"  - Precision:           {metrics['bert_score']['precision']:.4f}")
    print(f"  - Recall:              {metrics['bert_score']['recall']:.4f}")
    print(f"  - Model Used:          {metrics['bert_score'].get('model', 'unknown')}")
    print(f"Length Ratio:            {metrics['length_ratio']:.4f}")
    print(f"{'-'*80}")
    
    return {
        "timestamp": datetime.now().isoformat(),
        "config": asdict(config),
        "question": user_question,
        "response": response_text,
        "reference_answer": reference_answer,
        "prompt_sources": prompt_sources,
        "metrics": metrics,
        "tool_calls": tool_calls,
        "iterations": iterations,
        "bert_model": bert_model,
    }

print("✓ Evaluation function ready")

✓ Evaluation function ready


## Interactive Evaluation Studio

### Execute Multi-Model Evaluation

In [ ]:
# ============================================================================
# IMPROVED ANALYSIS - COMPARISON ACROSS MODELS
# ============================================================================

def analyze_se_results_detailed(results_dict):
    """
    Detailed analysis comparing models across all test cases.
    Shows per-test-case model comparison.
    """
    print(f"\n\n{'='*100}")
    print(f"📊 SE LEARNING EVALUATION - DETAILED ANALYSIS (ACROSS MODELS)")
    print(f"{'='*100}\n")
    
    # Overall average by model
    print(f"{'='*100}")
    print(f"📈 OVERALL AVERAGE METRICS BY MODEL")
    print(f"{'='*100}")
    print(f"{'Model':<40} {'BLEU':>12} {'ROUGE-L':>12} {'METEOR':>12} {'BERT-F1':>12} {'AVG':>12}")
    print(f"{'-'*100}")
    
    model_summaries = {}
    for model_name, results in results_dict.items():
        bleu_scores = [r['metrics']['bleu'] for r in results]
        rouge_scores = [r['metrics']['rouge_l'] for r in results]
        meteor_scores = [r['metrics']['meteor'] for r in results]
        bert_scores = [r['metrics']['bert_score']['f1'] for r in results]
        
        avg_bleu = np.mean(bleu_scores)
        avg_rouge = np.mean(rouge_scores)
        avg_meteor = np.mean(meteor_scores)
        avg_bert = np.mean(bert_scores)
        avg_overall = np.mean([avg_bleu, avg_rouge, avg_meteor, avg_bert])
        
        model_summaries[model_name] = {
            'bleu': avg_bleu,
            'rouge_l': avg_rouge,
            'meteor': avg_meteor,
            'bert': avg_bert,
            'overall': avg_overall,
            'all_results': results
        }
        
        print(f"{model_name:<40} {avg_bleu:>12.4f} {avg_rouge:>12.4f} {avg_meteor:>12.4f} {avg_bert:>12.4f} {avg_overall:>12.4f}")
    
    # Per-test-case comparison across models
    print(f"\n{'='*100}")
    print(f"📋 METRICS BY TEST CASE (ACROSS MODELS)")
    print(f"{'='*100}\n")
    
    for tc_idx, test_case in enumerate(SE_TEST_CASES):
        print(f"\n[Test Case #{test_case['id']}] {test_case['topic']}")
        print(f"Question: {test_case['question'][:80]}...")
        print(f"{'-'*100}")
        print(f"{'Model':<40} {'BLEU':>12} {'ROUGE-L':>12} {'METEOR':>12} {'BERT-F1':>12}")
        print(f"{'-'*100}")
        
        for model_name, results in results_dict.items():
            result = results[tc_idx]
            metrics = result['metrics']
            print(f"{model_name:<40} {metrics['bleu']:>12.4f} {metrics['rouge_l']:>12.4f} {metrics['meteor']:>12.4f} {metrics['bert_score']['f1']:>12.4f}")
        
        # Best model for this test case
        model_scores = {}
        for model_name, results in results_dict.items():
            result = results[tc_idx]
            metrics = result['metrics']
            avg_score = np.mean([metrics['bleu'], metrics['rouge_l'], metrics['meteor'], metrics['bert_score']['f1']])
            model_scores[model_name] = avg_score
        
        best_model = max(model_scores.items(), key=lambda x: x[1])
        print(f"\n→ Best Model: {best_model[0]} (avg: {best_model[1]:.4f})")
    
    # Overall rankings
    print(f"\n\n{'='*100}")
    print(f"🏆 OVERALL RANKINGS")
    print(f"{'='*100}\n")
    
    # Sort by overall score
    sorted_models = sorted(model_summaries.items(), key=lambda x: x[1]['overall'], reverse=True)
    
    print("By Overall Score (BLEU + ROUGE-L + METEOR + BERT-F1):")
    for rank, (model_name, summary) in enumerate(sorted_models, 1):
        print(f"  {rank}. {model_name:<45} {summary['overall']:.4f}")
    
    print("\nBy BERTScore (F1):")
    sorted_by_bert = sorted(model_summaries.items(), key=lambda x: x[1]['bert'], reverse=True)
    for rank, (model_name, summary) in enumerate(sorted_by_bert, 1):
        print(f"  {rank}. {model_name:<45} {summary['bert']:.4f}")
    
    print("\nBy BLEU:")
    sorted_by_bleu = sorted(model_summaries.items(), key=lambda x: x[1]['bleu'], reverse=True)
    for rank, (model_name, summary) in enumerate(sorted_by_bleu, 1):
        print(f"  {rank}. {model_name:<45} {summary['bleu']:.4f}")
    
    return model_summaries

print("✓ Detailed analysis function ready")
print(f"\nUsage: summaries = analyze_se_results_detailed(results)")

In [ ]:
# ============================================================================
# SE EVALUATION - EXECUTION GUIDE
# ============================================================================

guide = """
╔════════════════════════════════════════════════════════════════════════════╗
║                   SE LEARNING EVALUATION - QUICK START                    ║
╚════════════════════════════════════════════════════════════════════════════╝

📋 EVALUATION STRUCTURE:
  - 3 Models: meta-llama, qwen, microsoft/phi-3
  - 5 Test Cases: SDLC, Requirements, Process Models, Agile, Management
  - TOTAL: 3 models × 5 test cases = 15 evaluations
  
  Comparison: ACROSS MODELS (not temperatures)
  
  Test Case 1 (SDLC) → Evaluated by Model A, B, C
  Test Case 2 (Requirements) → Evaluated by Model A, B, C
  Test Case 3 (Process Models) → Evaluated by Model A, B, C
  Test Case 4 (Agile) → Evaluated by Model A, B, C
  Test Case 5 (Management) → Evaluated by Model A, B, C

🚀 EXECUTION FLOW:

  STEP 1: [OPTIONAL] Test with single model first
    → Run: "TEST SINGLE MODEL FIRST" cell
    → Run: "RUN SINGLE MODEL TEST" cell
    → Check: results look OK?
    
  STEP 2: [MAIN] Run multi-model evaluation
    → Run: "MULTI-MODEL EVALUATION - ALL TEST CASES" cell
    → Execute: results = await evaluate_se_multi_model()
    → Wait: ~15-30 mins (depending on model speed)
    
  STEP 3: [ANALYSIS] Analyze and compare results
    → Execute: summaries = analyze_se_results(results)
    → View: metrics comparison across all models
    
📊 SHARED CONFIGURATION FOR ALL EVALUATIONS:
  - System Prompt: SE Learning Assistant (comprehensive)
  - Learning Profile: Task + Persona + Mission
  - BERT Model: indobert-base-p1 (for Indonesian text)
  - Temperature: 0.7
  - Tools: Enabled (refine_prompt, semantic_search)
  - RAG: Enabled
  
💾 OUTPUT STRUCTURE:
  results = {
    "meta-llama/llama-3.1-8b-instruct": [result_tc1, result_tc2, ..., result_tc5],
    "qwen/qwen-2.5-7b-instruct": [result_tc1, result_tc2, ..., result_tc5],
    "microsoft/phi-3-medium-128k-instruct": [result_tc1, result_tc2, ..., result_tc5]
  }
  
  Where each result contains:
  - metrics: {bleu, rouge_l, meteor, bert_score}
  - response: LLM's answer
  - tool_calls: list of tools used
  - prompt_sources: which prompts were used
"""

print(guide)

### Quick Start Guide for SE Evaluation

In [ ]:
# ============================================================================
# ANALYZE & COMPARE MULTI-MODEL RESULTS
# ============================================================================

def analyze_se_results(results_dict):
    """
    Analyze and compare results across all models and test cases.
    """
    print(f"\n\n{'='*80}")
    print(f"📊 COMPREHENSIVE EVALUATION ANALYSIS")
    print(f"{'='*80}\n")
    
    # Per-model summary
    print(f"{'='*80}")
    print(f"📈 AVERAGE METRICS BY MODEL")
    print(f"{'='*80}")
    print(f"{'Model':<40} {'BLEU':>8} {'ROUGE-L':>8} {'METEOR':>8} {'BERT-F1':>8}")
    print(f"{'-'*80}")
    
    model_summaries = {}
    for model_name, results in results_dict.items():
        bleu_scores = [r['metrics']['bleu'] for r in results]
        rouge_scores = [r['metrics']['rouge_l'] for r in results]
        meteor_scores = [r['metrics']['meteor'] for r in results]
        bert_scores = [r['metrics']['bert_score']['f1'] for r in results]
        
        avg_bleu = np.mean(bleu_scores)
        avg_rouge = np.mean(rouge_scores)
        avg_meteor = np.mean(meteor_scores)
        avg_bert = np.mean(bert_scores)
        
        model_summaries[model_name] = {
            'bleu': avg_bleu,
            'rouge_l': avg_rouge,
            'meteor': avg_meteor,
            'bert': avg_bert,
            'overall': np.mean([avg_bleu, avg_rouge, avg_meteor, avg_bert])
        }
        
        print(f"{model_name:<40} {avg_bleu:>8.4f} {avg_rouge:>8.4f} {avg_meteor:>8.4f} {avg_bert:>8.4f}")
    
    # Per-test-case summary
    print(f"\n{'='*80}")
    print(f"📋 AVERAGE METRICS BY TEST CASE")
    print(f"{'='*80}")
    print(f"{'Test Case':<40} {'BLEU':>8} {'ROUGE-L':>8} {'METEOR':>8} {'BERT-F1':>8}")
    print(f"{'-'*80}")
    
    for i, test_case in enumerate(SE_TEST_CASES):
        bleu_scores = [results[i]['metrics']['bleu'] for results in results_dict.values()]
        rouge_scores = [results[i]['metrics']['rouge_l'] for results in results_dict.values()]
        meteor_scores = [results[i]['metrics']['meteor'] for results in results_dict.values()]
        bert_scores = [results[i]['metrics']['bert_score']['f1'] for results in results_dict.values()]
        
        print(f"{test_case['topic']:<40} {np.mean(bleu_scores):>8.4f} {np.mean(rouge_scores):>8.4f} {np.mean(meteor_scores):>8.4f} {np.mean(bert_scores):>8.4f}")
    
    # Best model
    print(f"\n{'='*80}")
    print(f"🏆 BEST MODELS")
    print(f"{'='*80}")
    best_overall = max(model_summaries.items(), key=lambda x: x[1]['overall'])
    best_bleu = max(model_summaries.items(), key=lambda x: x[1]['bleu'])
    best_bert = max(model_summaries.items(), key=lambda x: x[1]['bert'])
    
    print(f"Overall Best:    {best_overall[0]} ({best_overall[1]['overall']:.4f})")
    print(f"Best BLEU:       {best_bleu[0]} ({best_bleu[1]['bleu']:.4f})")
    print(f"Best BERTScore:  {best_bert[0]} ({best_bert[1]['bert']:.4f})")
    
    return model_summaries

print("✓ Analysis function ready")
print(f"\nUsage: summaries = analyze_se_results(results)")

In [ ]:
# ============================================================================
# MULTI-MODEL EVALUATION - ALL TEST CASES
# ============================================================================

# Models untuk evaluasi
SE_MODELS = [
    "meta-llama/llama-3.1-8b-instruct",
    "qwen/qwen-2.5-7b-instruct",
    "microsoft/phi-3-medium-128k-instruct"
]

async def evaluate_se_multi_model():
    """
    Evaluate all 3 models dengan semua test cases.
    Menggunakan system prompt, learning profile, dan BERT model yang sama untuk semua.
    """
    all_results = {}
    
    for model_name in SE_MODELS:
        print(f"\n\n{'#'*80}")
        print(f"# EVALUATING MODEL: {model_name}")
        print(f"{'#'*80}\n")
        
        model_results = []
        
        # Evaluate each test case
        for test_case in SE_TEST_CASES:
            print(f"\n{'='*80}")
            print(f"📝 Test Case #{test_case['id']}: {test_case['topic']}")
            print(f"{'='*80}")
            
            config = EvaluationConfig(
                model_name=model_name,
                temperature=0.7,
                use_tools=True,
                max_tool_iterations=3,
                prompt_general=SE_SYSTEM_PROMPT,
                task=SE_TASK,
                persona=SE_PERSONA,
                mission_objective=SE_MISSION,
                with_rag=True,
                top_k=5,
                similarity_threshold=0.7,
            )
            
            result = await run_evaluation(
                config=config,
                user_question=test_case['question'],
                reference_answer=test_case['reference_answer'],
                bert_model=BERT_MODEL,
                verbose=False  # Less verbose for batch evaluation
            )
            
            model_results.append(result)
            
            # Print summary for this test case
            print(f"✓ Test case complete")
            print(f"  BLEU:      {result['metrics']['bleu']:.4f}")
            print(f"  ROUGE-L:   {result['metrics']['rouge_l']:.4f}")
            print(f"  METEOR:    {result['metrics']['meteor']:.4f}")
            print(f"  BERTScore: {result['metrics']['bert_score']['f1']:.4f}")
        
        all_results[model_name] = model_results
    
    return all_results

print("✓ Multi-model evaluation function ready")
print(f"\n📋 Configuration:")
print(f"  Models: {len(SE_MODELS)}")
print(f"  Test Cases: {len(SE_TEST_CASES)}")
print(f"  Total Evaluations: {len(SE_MODELS) * len(SE_TEST_CASES)}")
print(f"  System Prompt: SE Learning Assistant (comprehensive)")
print(f"  BERT Model: {BERT_MODEL}")
print(f"\n⚠️  WARNING: This will take a while (multiple models × multiple test cases)")
print(f"\nTo run, execute: results = await evaluate_se_multi_model()")

### Multi-Model Evaluation (All Test Cases)

In [ ]:
# ============================================================================
# RUN SINGLE MODEL TEST (First test case)
# ============================================================================
# Jalankan dengan test case pertama dulu untuk verify setup

test_case = SE_TEST_CASES[0]  # Use first test case

print(f"\n{'='*80}")
print(f"🧪 SINGLE MODEL TEST - Test Case #{test_case['id']}: {test_case['topic']}")
print(f"{'='*80}\n")

single_model_result = await run_evaluation(
    config=SE_EVAL_CONFIG_SINGLE,
    user_question=test_case['question'],
    reference_answer=test_case['reference_answer'],
    bert_model=BERT_MODEL,
    verbose=True
)

print("\n✓ Single model test complete!")
print(f"\nMetrics Summary:")
print(f"  BLEU:      {single_model_result['metrics']['bleu']:.4f}")
print(f"  ROUGE-L:   {single_model_result['metrics']['rouge_l']:.4f}")
print(f"  METEOR:    {single_model_result['metrics']['meteor']:.4f}")
print(f"  BERTScore: {single_model_result['metrics']['bert_score']['f1']:.4f}")

In [ ]:
# ============================================================================
# TEST SINGLE MODEL FIRST
# ============================================================================
# Test dengan 1 model dulu untuk verify setup berjalan dengan baik

SE_EVAL_CONFIG_SINGLE = EvaluationConfig(
    # === LLM MODEL (Test with first model) ===
    model_name="meta-llama/llama-3.1-8b-instruct",  # Test dengan model pertama
    temperature=0.7,
    
    # === TOOL CONFIGURATION ===
    use_tools=True,
    max_tool_iterations=3,
    
    # === SYSTEM PROMPT ===
    prompt_general=SE_SYSTEM_PROMPT,  # Comprehensive system prompt with tools + guidelines
    
    # === LEARNING PROFILE ===
    task=SE_TASK,
    persona=SE_PERSONA,
    mission_objective=SE_MISSION,
    
    # === SPECIFIC PROMPT ===
    prompt_specific=None,  # Tidak ada specific prompt, sudah ada di system prompt
    
    # === RAG CONFIGURATION ===
    with_rag=True,
    top_k=5,
    similarity_threshold=0.7,
)

BERT_MODEL = "indobert-base-p1"

print("✓ Single Model Test Configuration ready")
print(f"  Model: {SE_EVAL_CONFIG_SINGLE.model_name}")
print(f"  Temperature: {SE_EVAL_CONFIG_SINGLE.temperature}")
print(f"  BERT Model: {BERT_MODEL}")
print(f"  Test Cases: {len(SE_TEST_CASES)}")
print(f"\n📝 Ready to test with first test case:")
print(f"   {SE_TEST_CASES[0]['question'][:80]}...")

### Single Model Test (Test First Model)

In [ ]:
# ============================================================================
# SE LEARNING - TEST CASES
# ============================================================================

SE_TEST_CASES = [
    {
        "id": 1,
        "question": "Aku masih agak bingung, sebenarnya SDLC itu apa sih? Kenapa di rekayasa perangkat lunak harus pakai SDLC?",
        "reference_answer": """SDLC atau Software Development Life Cycle adalah kerangka kerja yang digunakan untuk mengembangkan perangkat lunak secara sistematis dan terstruktur. SDLC membantu pengembang memahami tahapan apa saja yang perlu dilalui sejak masalah pertama kali diidentifikasi, kebutuhan pengguna dikumpulkan, sistem dirancang, perangkat lunak diimplementasikan, diuji, hingga akhirnya dipelihara setelah digunakan.

Penggunaan SDLC penting karena pengembangan perangkat lunak melibatkan banyak aktivitas, peran, dan keputusan. Tanpa kerangka kerja yang jelas, proyek perangkat lunak cenderung sulit dikendalikan, mudah mengalami perubahan yang tidak terkelola, serta berisiko menghasilkan sistem yang tidak sesuai kebutuhan pengguna. Dengan SDLC, proses pengembangan menjadi lebih terencana, terukur, dan hasilnya lebih dapat dipertanggungjawabkan.""",
        "topic": "SDLC Overview"
    },
    {
        "id": 2,
        "question": "Di tahap requirements engineering itu biasanya ngapain aja? Terus output atau dokumen yang dihasilkan apa?",
        "reference_answer": """Tahap requirements engineering berfokus pada memahami dan mendefinisikan apa yang dibutuhkan oleh pengguna dan stakeholder dari sebuah sistem. Aktivitas di tahap ini biasanya meliputi pengumpulan kebutuhan (misalnya melalui wawancara, observasi, atau diskusi), analisis kebutuhan untuk memastikan tidak ada konflik atau ambiguitas, pendokumentasian kebutuhan secara sistematis, serta validasi kebutuhan bersama pengguna.

Hasil utama dari tahap ini adalah dokumen kebutuhan, yang paling umum dikenal sebagai Software Requirements Specification (SRS). Dokumen ini menjadi dasar bagi tahap desain dan implementasi. Kesalahan atau ketidakjelasan pada tahap requirements sering kali menyebabkan kegagalan proyek di tahap-tahap selanjutnya.""",
        "topic": "Requirements Engineering"
    },
    {
        "id": 3,
        "question": "Kalau buat proyek tugas kelompok yang kecil, lebih cocok pakai Waterfall atau Agile? Kenapa?",
        "reference_answer": """Untuk proyek tugas kelompok yang relatif kecil dan memiliki kebutuhan yang masih bisa berubah, pendekatan Agile umumnya lebih sesuai dibandingkan Waterfall. Agile memungkinkan tim bekerja secara iteratif dan bertahap, sehingga perubahan kebutuhan dapat ditangani dengan lebih fleksibel.

Sebaliknya, model Waterfall lebih cocok untuk proyek dengan kebutuhan yang sudah jelas dan stabil sejak awal. Dalam konteks tugas kuliah, kebutuhan sering kali berkembang seiring diskusi dan pemahaman tim, sehingga Agile lebih realistis dan lebih mendukung kolaborasi mahasiswa.""",
        "topic": "Process Models Comparison"
    },
    {
        "id": 4,
        "question": "Agile itu sebenarnya metode atau model sih? Aku sering ketuker soalnya.",
        "reference_answer": """Agile bukan satu metode atau satu model proses tertentu, melainkan sebuah pendekatan atau filosofi dalam pengembangan perangkat lunak. Agile menekankan nilai dan prinsip seperti kolaborasi tim, respons terhadap perubahan, dan pengiriman perangkat lunak secara bertahap.

Dari pendekatan Agile ini kemudian muncul berbagai metode konkret, seperti Scrum, Extreme Programming (XP), dan Kanban. Dengan kata lain, Agile adalah payung besarnya, sedangkan Scrum atau XP adalah metode yang bisa diterapkan secara langsung dalam proyek.""",
        "topic": "Agile vs Methods"
    },
    {
        "id": 5,
        "question": "Di manajemen proyek software, sebenarnya peran manajer proyek itu ngapain aja? Apakah cuma ngatur jadwal?",
        "reference_answer": """Manajer proyek perangkat lunak tidak hanya bertugas mengatur jadwal. Perannya mencakup perencanaan proyek, estimasi biaya dan waktu, pengelolaan risiko, pengendalian kualitas, serta menjaga komunikasi dan koordinasi antar anggota tim dan stakeholder.

Dalam proyek perangkat lunak, kegagalan sering terjadi bukan karena masalah teknis semata, tetapi karena kurangnya pengelolaan yang baik. Oleh karena itu, manajemen proyek menjadi komponen penting dalam keberhasilan pengembangan perangkat lunak, termasuk dalam skala proyek akademik.""",
        "topic": "Project Management Role"
    }
]

print(f"✓ {len(SE_TEST_CASES)} test cases loaded")
for tc in SE_TEST_CASES:
    print(f"  [{tc['id']}] {tc['topic']}: {tc['question'][:60]}...")

### Test Cases for SE Learning Evaluation

In [ ]:
# ============================================================================
# SE LEARNING PROFILE
# ============================================================================

SE_TASK = """Saya mahasiswa S1 yang lagi belajar mata kuliah Proses Rekayasa Perangkat Lunak. 
Saya pengen belajar secara bertahap dan pelan-pelan tentang Software Development Life Cycle (SDLC), 
mulai dari:
- cara mengidentifikasi masalah dan kebutuhan pengguna (requirements engineering),
- perancangan sistem dan software,
- proses implementasi,
- pengujian dan quality assurance,
- sampai ke pemeliharaan sistem.

Saya juga pengen ngerti alur besar prosesnya, 
kenapa tiap tahap itu penting, 
dan output atau artefak apa yang biasanya dihasilkan di setiap tahap."""

SE_PERSONA = """Saya ingin kamu berperan sebagai dosen pembimbing sekaligus tutor belajar yang sabar dan komunikatif untuk mahasiswa sarjana seperti saya

Bantu saya jelasin materi pakai bahasa yang santai tapi tetap jelas,
nggak terlalu teoritis, dan kalau bisa pakai contoh 
(seperti tugas kelompok, proyek kampus, atau studi kasus sederhana).

Kalau ada kesalahan yang sering dilakukan mahasiswa, tolong kasih tahu dan jelaskan pelan-pelan supaya saya nggak salah paham."""

SE_MISSION = """Target saya setelah belajar di sistem pembelajaran ini adalah:
- saya bisa paham proses rekayasa perangkat lunak secara utuh,
- mengerti metode dan model pengembangan perangkat lunak,
- saya punya gambaran dasar soal manajemen proyek perangkat lunak.

Saya juga pengen bisa:
- jelasin tiap tahap SDLC dengan kata-kata saya sendiri,
- tahu harus ngapain di tiap tahapan,
- dan saya bisa menerapkan konsepnya ke tugas kuliah atau proyek kelompok.

Kalau bisa, bantu saya juga buat mengerti:
dokumen apa yang perlu disiapkan pada tiap tahapan dan metode apa yang paling cocok buat kondisi tertentu."""

print("✓ SE Learning Profile loaded")
print(f"\n📚 Task: {SE_TASK[:80]}...")
print(f"👤 Persona: {SE_PERSONA[:80]}...")
print(f"🎯 Mission: {SE_MISSION[:80]}...")

### Learning Profile

In [ ]:
# ============================================================================
# SE LEARNING ASSISTANT - SYSTEM PROMPT (Combined)
# ============================================================================

SE_SYSTEM_PROMPT = """You are a helpful learning assistant with access to two powerful tools:

## Available Tools:
1. **refine_prompt** - Use this tool to clarify or improve unclear/ambiguous student questions
   - When: The student's question is vague, incomplete, or could be interpreted multiple ways
   - Result: Returns a more specific and clear version of the question for better search results
   - Example: "gimana loop?" → "Jelaskan apa itu loop dan perbedaan antara for dan while loop"

2. **semantic_search** - Use this tool to find relevant documents and information from the knowledge base
   - When: You need to retrieve specific information to answer the student's question
   - Input: A clear search query (ideally refined from refine_prompt if needed)
   - Result: Returns relevant document chunks with page numbers and similarity scores

## Recommended Workflow:
1. **Read the student's question carefully**
   - If it's unclear/ambiguous → Use refine_prompt to clarify first
   - If it's already clear → Proceed to semantic_search
2. **Search the knowledge base**
   - Use the refined (or original if clear) question to search documents
   - Pass it as the query parameter to semantic_search
3. **Synthesize and answer**
   - Combine the search results with your knowledge
   - Cite the document sources when appropriate
   - Provide comprehensive answer that addresses the student's actual need

## SE Learning Assistant Role:
You are an interactive learning assistant designed specifically to help undergraduate students (S1) learn Software Engineering.
Your main focus areas are:
1) Software Engineering Process
2) Software Engineering Methods and Models
3) Software Engineering Management

Your role is NOT to act like a lecturer who gives long one-way explanations, but as a learning companion that:
- Mengajak mahasiswa untuk berfikir
- Membantu mahasiswa memahami materi pembelajaran pelan-pelan
- Mengkaitkan teori dengan contoh dunia nyata dan tugas kuliah

## Communication Style & Behaviour:
- Use casual, friendly, and natural Indonesian (bahasa sehari-hari mahasiswa), not rigid academic language
- Avoid overly formal sentences, but keep explanations conceptually correct
- Assume the student is still learning and may be confused or unsure
- If the student asks something vague atau terlalu umum, guide them dengan pertanyaan balik yang membantu memperjelas
- Maintain friendly, human-like interaction
- No overly rigid academic tone

## Learning Strategy:
- Start from the student's current understanding
- Break down complex concepts into simple steps
- Use analogies, daily-life examples, or project-based scenarios (contoh: tugas kelompok, proyek capstone, skripsi, tugas akhir)
- When relevant, relate concepts to SDLC phases: requirements, design, implementation, testing, deployment, maintenance

## SE Process Guideline:
- Explain concepts like SDLC, process models, activities, roles, artifacts, and workflows
- Compare process models (Waterfall, Agile, Spiral, Incremental, DevOps) using clear strengths, weaknesses, and use cases
- Emphasize "kenapa proses itu penting berikan penjelasan", bukan cuma "apa definisinya"
- When discussing process models:
  - Explain context of use
  - Strengths and weaknesses
  - Typical mistakes students make when choosing them

## SE Method and Model Guideline:
- Explain methods (e.g., Agile methods, Scrum, XP, Kanban) and modeling techniques (UML, BPMN, etc.) at a conceptual level
- Focus on how and when a method/model is used, not just notation
- If diagrams are mentioned, describe them verbally (do not assume drawing tools)
- Explain methods (Scrum, XP, Kanban, etc.) as practical approaches, not buzzwords
- Explain models and notations conceptually (e.g., UML diagrams without assuming drawing)
- Focus on decision logic: "kapan metode ini cocok, kapan tidak"

## SE Management Guideline:
- Cover topics like project planning, estimation, scheduling, risk management, quality management, and team coordination
- Use realistic scenarios: tugas kelompok, proyek kampus, sistem informasi skala kecil–menengah
- Highlight trade-offs and management decisions, not idealized theory

## Interaction Rules:
- Ask reflective questions when appropriate (e.g., "kalau menurut kamu…", "di kasus ini, model apa yang cocok?")
- Encourage critical thinking instead of giving final answers immediately
- If the student asks for "jawaban langsung", you may give it, but also explain the reasoning
- Treat it as a learning intention, not just a question
- Assume the student may be unsure, partially correct, or exploring ideas
- Do NOT directly jump to long explanations unless clearly requested
- Identify what the student is trying to learn:
  - concept understanding
  - comparison of methods/models
  - problem-solving in a project context
  - exam or assignment preparation

If the input is vague or ambiguous:
- Ask short clarifying questions using casual, student-friendly language
- Example: "Ini lagi bahas konsep doang, atau mau dipakai buat tugas/proyek?"

## Student Profiling & Adaptation:
You will analyze the student's level of understanding and adapt your explanations accordingly:
- **Dasar (Beginner)**:
  - Use simple language
  - Step-by-step explanations
  - Concrete examples
- **Menengah (Intermediate)**:
  - Use comparisons
  - Highlight trade-offs
  - Introduce reasoning and decision logic
- **Lanjut (Advanced)**:
  - Use analytical tone
  - Discuss limitations, assumptions, and real-world constraints

## Assessment & Feedback:
- Help students understand assignments, quizzes, or exam preparation
- When asked, generate practice questions, mini-cases, or simple exercises
- Give feedback in a supportive tone, not judgmental
- After delivering an explanation, internally assess whether the student likely understands
- If understanding seems low, re-explain using a different angle or analogy
- If understanding seems sufficient, provide a short summary

## Important Limitations:
- Do not invent references or claim using specific textbooks unless explicitly provided
- Do not assume advanced industry experience from the student
- Stay within Software Engineering domain; avoid unrelated topics unless needed as analogy
- If the system provides retrieved knowledge from documents, integrate naturally without mentioning retrieval mechanisms

## Key Principle:
Always prioritize understanding what the student actually needs. Use tools strategically. Reference document sources to build trust."""

print("✓ SE System Prompt loaded")
print(f"  Length: {len(SE_SYSTEM_PROMPT)} characters")

### System Prompt (General) - SE Learning Assistant

## SE Learning Evaluation Setup

### Step 1: Configure Evaluation

In [17]:
# ============================================================================
# CONFIGURATION INPUT SECTION
# ============================================================================
# Ubah nilai-nilai di bawah sesuai kebutuhan evaluasi Anda

EVAL_CONFIG = EvaluationConfig(
    # === LLM MODEL SELECTION ===
    model_name="gpt-4.1-nano",  # Pilih dari available models di database
    # Contoh: "claude-haiku-4-5", "gemini-2.5-flash", dll
    temperature=0.5,  # Range: 0.0 (deterministic) - 1.0 (creative)
    
    # === TOOL CONFIGURATION ===
    use_tools=True,  # Enable tool calling (refine_prompt, semantic_search)
    max_tool_iterations=3,  # Max agentic loop iterations
    
    # === SYSTEM PROMPT LAYERS (Priority 1 - HIGHEST) ===
    prompt_general=None,  # Custom system prompt (optional)
    # Contoh:
    # prompt_general="""Anda adalah asisten pembelajaran ahli. Berikan jawaban komprehensif 
    #                    dengan contoh praktis dan aplikasi dunia nyata."""
    
    # === LEARNING PROFILE (Priority 2 - MID) ===
    task="Learn about machine learning fundamentals",  # Apa yang ingin dipelajari?
    persona="Beginner programmer with basic Python knowledge",  # Profil learner
    mission_objective="Master neural networks and deep learning concepts",  # Tujuan jangka panjang
    
    # === SPECIFIC PROMPT (Priority 3 - LOWEST) ===
    # BARU: Bisa input langsung atau dari database
    prompt_specific="""Jelaskan dengan cara yang mudah dipahami oleh pemula. 
Gunakan analogi dan contoh dari kehidupan sehari-hari.""",  # Direct input (BARU)
    # prompt_id=None,  # Atau gunakan prompt dari database dengan ID
    
    # === RAG CONFIGURATION ===
    with_rag=True,  # Enable semantic search
    top_k=5,  # Number of documents to retrieve
    similarity_threshold=0.7,  # Min similarity score (0.0 - 1.0)
)

# === BERT MODEL SELECTION ===
BERT_MODEL = "indobenchmark/indobert-base-p1"  # BARU: Change untuk evaluate Indonesian text
# Opsi:
# - "indobert-base-p1"      (RECOMMENDED untuk Indonesian)
# - "indobert-large-p1"     (Lebih powerful, lebih lambat)
# - "distilbert-base-uncased" (Fallback untuk English)

print("✓ Configuration loaded")
print(f"  LLM Model: {EVAL_CONFIG.model_name}")
print(f"  Temperature: {EVAL_CONFIG.temperature}")
print(f"  Tools: {EVAL_CONFIG.use_tools}")
print(f"  BERT Model: {BERT_MODEL}")
print(f"\n  Learning Profile:")
print(f"    - Task: {EVAL_CONFIG.task[:50]}...")
print(f"    - Persona: {EVAL_CONFIG.persona[:50]}...")
print(f"\n  System Prompts:")
print(f"    - General: {EVAL_CONFIG.prompt_general[:50] if EVAL_CONFIG.prompt_general else 'None'}...")
print(f"    - Specific: {EVAL_CONFIG.prompt_specific[:50] if EVAL_CONFIG.prompt_specific else 'None'}...")

✓ Configuration loaded
  LLM Model: gpt-4.1-nano
  Temperature: 0.5
  Tools: True
  BERT Model: indobenchmark/indobert-base-p1

  Learning Profile:
    - Task: Learn about machine learning fundamentals...
    - Persona: Beginner programmer with basic Python knowledge...

  System Prompts:
    - General: None...
    - Specific: Jelaskan dengan cara yang mudah dipahami oleh pemu...


### Step 2: Input Question & Reference Answer

In [18]:
# ============================================================================
# QUESTION & REFERENCE ANSWER INPUT
# ============================================================================
# Ubah pertanyaan dan referensi jawaban sesuai yang ingin dievaluasi

USER_QUESTION = "gimana machine learning itu?"
# Pertanyaan yang akan dikirim ke LLM

REFERENCE_ANSWER = """Machine learning is a field of artificial intelligence that enables systems 
to learn from data without being explicitly programmed. It uses algorithms and statistical models 
to identify patterns in data and make predictions or decisions based on those patterns."""
# Jawaban referensi untuk perbandingan metrik

print("✓ Question and Reference Answer loaded")
print(f"\n  Question length: {len(USER_QUESTION)} chars")
print(f"  Reference answer length: {len(REFERENCE_ANSWER)} chars")

✓ Question and Reference Answer loaded

  Question length: 28 chars
  Reference answer length: 263 chars


### Step 3: Run Evaluation

In [19]:
# Run the evaluation with selected BERT model
result = await run_evaluation(
    config=EVAL_CONFIG,
    user_question=USER_QUESTION,
    reference_answer=REFERENCE_ANSWER,
    bert_model=BERT_MODEL,  # Pass BERT model selection
    verbose=True
)

print("\n✓ Evaluation complete!")


🧪 NLP EVALUATION RUN
Model: gpt-4.1-nano
With Tools: True
Temperature: 0.5
Max Iterations: 3
BERT Model: indobenchmark/indobert-base-p1
--------------------------------------------------------------------------------

📋 SYSTEM PROMPT SOURCES:
  - prompt_general: You are a helpful learning assistant with access to two powerful tools:\r                           ...
  - task: Learn about machine learning fundamentals
  - persona: Beginner programmer with basic Python knowledge
  - mission_objective: Master neural networks and deep learning concepts
  - specific_prompt: Jelaskan dengan cara yang mudah dipahami oleh pemula. 
Gunakan analogi dan contoh dari kehidupan seh...
  - specific_prompt_source: Direct input

📝 ASSEMBLED SYSTEM PROMPT:
--------------------------------------------------------------------------------
# General Prompt
You are a helpful learning assistant with access to two powerful tools:\r                                  +

## Student Learning Profile
# Task
Learn abo

In [20]:
# Display detailed results
print("\n" + "="*80)
print("📊 DETAILED EVALUATION RESULTS")
print("="*80)

metrics = result['metrics']

print(f"\n🎯 METRIC SCORES:")
print(f"  BLEU:           {metrics['bleu']:.4f} (0-1 scale)")
print(f"  ROUGE-L:        {metrics['rouge_l']:.4f} (0-1 scale)")
print(f"  METEOR:         {metrics['meteor']:.4f} (0-1 scale)")
print(f"  BERTScore-F1:   {metrics['bert_score']['f1']:.4f} (0-1 scale)")
print(f"\n  → Average Score: {np.mean([metrics['bleu'], metrics['rouge_l'], metrics['meteor'], metrics['bert_score']['f1']]):.4f}")

print(f"\n📏 RESPONSE CHARACTERISTICS:")
print(f"  Length Ratio:   {metrics['length_ratio']:.2f}x (vs reference)")
ref_len = len(REFERENCE_ANSWER.split())
resp_len = len(result['response'].split())
print(f"  Reference length: {ref_len} words")
print(f"  Response length:  {resp_len} words")

print(f"\n🔧 TOOL EXECUTION:")
print(f"  Tool calls made: {len(result['tool_calls'])}")
for i, tc in enumerate(result['tool_calls'], 1):
    print(f"    [{i}] {tc['name']}")
    if tc['name'] == 'refine_prompt' and tc['result'].get('success'):
        print(f"        → Refined: {tc['result'].get('refined', '')[:60]}...")
    elif tc['name'] == 'semantic_search':
        print(f"        → Found {len(tc['result'].get('results', []))} documents")

print(f"\n💾 SAVED RESULT OBJECT: 'result'")
print(f"   - To save results to file: run next cell")


📊 DETAILED EVALUATION RESULTS

🎯 METRIC SCORES:
  BLEU:           0.0031 (0-1 scale)
  ROUGE-L:        0.0543 (0-1 scale)
  METEOR:         0.0908 (0-1 scale)
  BERTScore-F1:   0.5504 (0-1 scale)

  → Average Score: 0.1747

📏 RESPONSE CHARACTERISTICS:
  Length Ratio:   3.62x (vs reference)
  Reference length: 39 words
  Response length:  141 words

🔧 TOOL EXECUTION:
  Tool calls made: 2
    [1] refine_prompt
        → Refined: RAJA, tolong jelaskan secara sederhana apa itu machine learn...
    [2] semantic_search
        → Found 5 documents

💾 SAVED RESULT OBJECT: 'result'
   - To save results to file: run next cell


### Step 5: Save Results (Optional)

In [13]:
# Save results to JSON file
import json

filename = f"evaluation_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

# Convert result to JSON-serializable format
result_json = {
    **result,
    "bert_score": result['metrics']['bert_score'],  # Already serializable
}

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(result_json, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {filename}")

✓ Results saved to: evaluation_result_20260102_192712.json


## Helper Functions for Experimentation

### Run Multiple Evaluations with Different Configs

In [14]:
async def run_multiple_evaluations(
    configs: List[EvaluationConfig],
    user_question: str,
    reference_answer: str
) -> List[Dict[str, Any]]:
    """
    Run evaluation with multiple configurations and compare results.
    Useful for comparing different models, temperatures, etc.
    """
    results = []
    
    for i, config in enumerate(configs, 1):
        print(f"\n{'#'*80}")
        print(f"# EVALUATION {i}/{len(configs)}: {config.model_name}")
        print(f"{'#'*80}")
        
        result = await run_evaluation(
            config=config,
            user_question=user_question,
            reference_answer=reference_answer,
            verbose=False  # Less verbose for multiple runs
        )
        results.append(result)
    
    # Compare metrics
    print(f"\n\n{'='*80}")
    print(f"📊 COMPARISON SUMMARY")
    print(f"{'='*80}")
    print(f"{'Model':<30} {'BLEU':>8} {'ROUGE-L':>8} {'METEOR':>8} {'BERT-F1':>8}")
    print(f"{'-'*80}")
    
    for result in results:
        model = result['config']['model_name']
        metrics = result['metrics']
        print(f"{model:<30} {metrics['bleu']:>8.4f} {metrics['rouge_l']:>8.4f} {metrics['meteor']:>8.4f} {metrics['bert_score']['f1']:>8.4f}")
    
    return results

print("✓ Helper function loaded: run_multiple_evaluations")
print("\nExample usage:")
print("""configs = [
    EvaluationConfig(model_name="gpt-4.1-nano", temperature=0.7),
    EvaluationConfig(model_name="claude-haiku-4-5", temperature=0.7),
    EvaluationConfig(model_name="gemini-2.5-flash", temperature=0.7),
]
results = await run_multiple_evaluations(configs, USER_QUESTION, REFERENCE_ANSWER)
""")

✓ Helper function loaded: run_multiple_evaluations

Example usage:
configs = [
    EvaluationConfig(model_name="gpt-4.1-nano", temperature=0.7),
    EvaluationConfig(model_name="claude-haiku-4-5", temperature=0.7),
    EvaluationConfig(model_name="gemini-2.5-flash", temperature=0.7),
]
results = await run_multiple_evaluations(configs, USER_QUESTION, REFERENCE_ANSWER)



In [ ]:
async def run_multiple_evaluations(
    configs: List[EvaluationConfig],
    user_question: str,
    reference_answer: str,
    bert_model: str = "indobert-base-p1"
) -> List[Dict[str, Any]]:
    """
    Run evaluation with multiple configurations and compare results.
    Useful for comparing different models, temperatures, etc.
    
    Args:
        configs: List of EvaluationConfig to test
        user_question: User's question
        reference_answer: Reference answer
        bert_model: BERT model to use for all evaluations
    """
    results = []
    
    for i, config in enumerate(configs, 1):
        print(f"\n{'#'*80}")
        print(f"# EVALUATION {i}/{len(configs)}: {config.model_name}")
        print(f"{'#'*80}")
        
        result = await run_evaluation(
            config=config,
            user_question=user_question,
            reference_answer=reference_answer,
            bert_model=bert_model,
            verbose=False  # Less verbose for multiple runs
        )
        results.append(result)
    
    # Compare metrics
    print(f"\n\n{'='*80}")
    print(f"📊 COMPARISON SUMMARY")
    print(f"{'='*80}")
    print(f"BERT Model Used: {bert_model}")
    print(f"{'='*80}")
    print(f"{'Model':<30} {'BLEU':>8} {'ROUGE-L':>8} {'METEOR':>8} {'BERT-F1':>8}")
    print(f"{'-'*80}")
    
    for result in results:
        model = result['config']['model_name']
        metrics = result['metrics']
        print(f"{model:<30} {metrics['bleu']:>8.4f} {metrics['rouge_l']:>8.4f} {metrics['meteor']:>8.4f} {metrics['bert_score']['f1']:>8.4f}")
    
    return results

print("✓ Helper function loaded: run_multiple_evaluations")
print("\nExample usage:")
print("""
configs = [
    EvaluationConfig(model_name="gpt-4.1-nano", temperature=0.7),
    EvaluationConfig(model_name="claude-haiku-4-5", temperature=0.7),
    EvaluationConfig(model_name="gemini-2.5-flash", temperature=0.7),
]
results = await run_multiple_evaluations(
    configs, 
    USER_QUESTION, 
    REFERENCE_ANSWER,
    bert_model="indobert-base-p1"
)
""")

In [15]:
async def compare_temperatures(
    model_name: str,
    temperatures: List[float],
    user_question: str,
    reference_answer: str,
    bert_model: str = "indobert-base-p1"
) -> List[Dict[str, Any]]:
    """
    Compare evaluation results across different temperature settings.
    
    Args:
        model_name: LLM model to use
        temperatures: List of temperature values to test
        user_question: User's question
        reference_answer: Reference answer
        bert_model: BERT model to use for evaluation
    """
    configs = [
        EvaluationConfig(
            model_name=model_name,
            temperature=temp,
            task="Learn about machine learning fundamentals",
            persona="Beginner programmer with basic Python knowledge",
            mission_objective="Master neural networks and deep learning concepts"
        )
        for temp in temperatures
    ]
    
    return await run_multiple_evaluations(configs, user_question, reference_answer, bert_model=bert_model)

print("✓ Helper function loaded: compare_temperatures")
print("\nExample usage:")
print("""
results = await compare_temperatures(
    model_name="gpt-4.1-nano",
    temperatures=[0.3, 0.7, 1.0],
    user_question=USER_QUESTION,
    reference_answer=REFERENCE_ANSWER,
    bert_model="indobert-base-p1"
)
""")

✓ Helper function loaded: compare_temperatures

Example usage:

results = await compare_temperatures(
    model_name="gpt-4.1-nano",
    temperatures=[0.3, 0.7, 1.0],
    user_question=USER_QUESTION,
    reference_answer=REFERENCE_ANSWER,
    bert_model="indobert-base-p1"
)

